# Runner — Anatomy-Aware Pose Estimation
Notebook **minimale**: tutto il codice vero sta nei moduli `.py` sul repo GitHub.
Qui dentro si fa solo: clone del repo → import → dati → training → valutazione.

**Prima di lanciare:**
1. Settings (pannello a destra) → **Internet: On** (serve per il `git clone`).
2. **Add Input** → aggiungi *COCO 2017 Keypoints* e il dataset *OCHuman* (condiviso).

In [ ]:
# === 1. Codice dal repo GitHub (single source of truth) ===
import os, sys

REPO_URL = "https://github.com/flaviomassaroni/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks.git"
REPO_DIR = "/kaggle/working/repo"

!rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}

# trova la cartella che contiene config.py e mettila nel path
mod_dir = None
for root, dirs, files in os.walk(REPO_DIR):
    if ".git" in root:
        continue
    if "config.py" in files:
        mod_dir = root
        break

if mod_dir:
    sys.path.insert(0, mod_dir)
    print("Moduli trovati in:", mod_dir)
    print("File .py:", [f for f in os.listdir(mod_dir) if f.endswith(".py")])
else:
    print("config.py NON trovato.")
    print("Contenuto repo:", os.listdir(REPO_DIR) if os.path.exists(REPO_DIR) else "clone fallito (repo privata?)")

In [ ]:
# === 2. Import dei moduli + seed + check dataset montati ===
from config import *
import utils, data, network, train
import evaluation as ev

set_seed(SEED)
print("Device:", device)

print("\n/kaggle/input contiene:")
for d in os.listdir("/kaggle/input"):
    print("  -", d)

In [ ]:
# === 3. Dati COCO ===
from torch.utils.data import DataLoader

train_samples = data.build_samples(COCO_TRAIN_ANN, min_keypoints=8)
val_samples   = data.build_samples(COCO_VAL_ANN)
print(f"train: {len(train_samples)} | val: {len(val_samples)}")

train_dataset = data.COCOKeypointsDataset(train_samples, COCO_TRAIN_IMG, INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)
val_dataset   = data.COCOKeypointsDataset(val_samples,   COCO_VAL_IMG,   INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# === 4. Training (lanciare di sera — saltare se si usa un best.pth gia' addestrato) ===
import torch

NUM_EPOCHS = 30
model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[15, 25], gamma=0.1)
criterion = train.WeightedMSELoss()

history = train.fit(model, train_loader, val_loader, optimizer, scheduler, criterion, device, NUM_EPOCHS, CHECKPOINT_DIR, resume=True)

In [ ]:
# === 5. Valutazione: COCO val + OCHuman zero-shot ===
import torch

# Se hai saltato la cella 4 e monti il best.pth da un altro notebook:
# 1. Aggiungi l'output della versione come Input
# 2. Trova il path:  !find /kaggle/input -name 'best.pth'
# 3. Sostituisci la riga sotto col path trovato
BEST_PTH = f'{CHECKPOINT_DIR}/best.pth'

model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(BEST_PTH, map_location=device))

coco_eval, avr_coco = ev.evaluate_on_coco_val(model, val_samples, device)
oc_eval,   avr_oc   = ev.evaluate_on_ochuman(model, device)

LABELS = ["AP","AP.50","AP.75","AP_M","AP_L","AR","AR.50","AR.75","AR_M","AR_L"]
print(f"\n{'metrica':<10}{'COCO val':>12}{'OCHuman':>12}")
print("-"*34)
for i, lab in enumerate(LABELS):
    print(f"{lab:<10}{coco_eval.stats[i]:>12.4f}{oc_eval.stats[i]:>12.4f}")
print(f"\n{'AVR rate':<10}{avr_coco['AVR_pose_rate']:>12.4f}{avr_oc['AVR_pose_rate']:>12.4f}")
print(f"{'AVR mean':<10}{avr_coco['AVR_mean_violations']:>12.4f}{avr_oc['AVR_mean_violations']:>12.4f}")